# CS-162 Cartography — Colab Training Suite

1. **Runtime → Change runtime type → T4 GPU** → **Restart session**
2. Run **clone + GPU check** (must show `cuda=True` **before** pip installs)
3. Credentials + `colab_setup.sh` (uses `requirements-colab.txt` — does **not** install CPU torch)
4. Training cells: DistilBERT smoke → Llama / Ministral mini tests → full suite

CUDA help: [docs/COLAB_CUDA.md](../docs/COLAB_CUDA.md)

In [1]:
# Run AFTER: Runtime → T4 GPU → Restart session
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/aditya-am-murthy/CS-162-Final.git"
REPO_DIR = Path("/content/CS-162-Final")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

# GPU must exist before any pip install
!nvidia-smi
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA — enable T4 GPU and Restart session, then re-run this cell")
print("GPU:", torch.cuda.get_device_name(0))

working directory: /content/CS-162-Final
/bin/bash: line 1: nvidia-smi: command not found
torch: 2.10.0+cpu | cuda: False


RuntimeError: No CUDA — enable T4 GPU and Restart session, then re-run this cell

### Credentials (Colab secrets, or getpass below — never commit tokens)

In [2]:
import os
from getpass import getpass

hf = os.environ.get("HF_TOKEN") or getpass("HF token (hf_...): ")
wandb_key = os.environ.get("WANDB_API_KEY") or getpass("W&B API key (optional, Enter to skip): ")

In [ ]:
from pathlib import Path

Path("hf_credentials.txt").write_text(f"hf_token={hf}\n", encoding="utf-8")
if wandb_key.strip():
    Path("wandb_credentials.txt").write_text(
        f"api_key={wandb_key}\nproject=cs162-dataset-cartography\n",
        encoding="utf-8",
    )

# Uses requirements-colab.txt (no PyPI torch). Re-checks CUDA after unsloth.
!bash scripts/colab_setup.sh
!python scripts/check_cuda.py --label notebook_after_setup

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 9.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 10

### Smoke test — DistilBERT (~5–15 min)

In [ ]:
!python scripts/run_cartography_experiment.py --task snli --preset distilbert \
  --max-train-samples 500 --epochs 2 --wandb-run-name colab_smoke

### Mini tests — Llama 3.2 & Ministral 3 (~10–25 min each on T4)

Run these **before** the full `train_all_models` suite. They use 200 train / 200 val examples and **1 epoch** to catch pad-token / Unsloth / CUDA issues early.

In [ ]:
# Llama 3.2 1B — mini (needs Meta license on HF + pad_token fix)
!python scripts/run_cartography_experiment.py --task dynamic --preset llama-3.2-1b \
  --max-train-samples 200 --max-eval-samples 200 --epochs 1 \
  --curriculum-after-epoch 0 --wandb-run-name colab_mini_llama --no-publish

In [ ]:
# Ministral 3 3B (Unsloth 4-bit) — mini
!python scripts/run_cartography_experiment.py --task dynamic --preset ministral-3b \
  --max-train-samples 200 --max-eval-samples 200 --epochs 1 \
  --curriculum-after-epoch 0 --wandb-run-name colab_mini_ministral --no-publish

### Full training (overnight on T4)

Run only after smoke + mini tests pass.

In [ ]:
!python scripts/train_all_models.py --max-train-samples 10000 --epochs 5

In [ ]:
# Idea #1: Preference Data Maps
!python scripts/run_cartography_experiment.py --task preference --preset distilbert \
  --max-train-samples 3000 --epochs 5 --wandb-run-name colab_preference

In [ ]:
# Idea #2: dynamic snapshots + curriculum
!python scripts/run_cartography_experiment.py --task dynamic --preset roberta-base \
  --curriculum-after-epoch 2 --epochs 5 --wandb-run-name colab_dynamic